# Lab 3 — Groups Nobody Labelled

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INCORTX/INCORTX.github.io/blob/master/DataAnalytics/session-03/lab_03.ipynb)

**Data Analytics · Lab session 3**

Two sessions of showing the model the right answer. **This session there is no answer** — no label,
no number, only rows — and the model has to find the groups on its own. That changes the hard question:
not *how close was it*, but *did it find anything real at all*.

**The demo is one story in two acts.** Act one runs k-Means from Lecture 5 (Unsupervised Learning) on
342 penguins and gets three clean groups — and just as clean a picture for k = 2, 4, 5 and 6. Act two
looks at what those clean pictures hid: **k was your choice**, not the data's; a hidden answer key says
the first answer was **68% right**; and putting every column on the same scale takes that to **92%**
while the score you would have trusted goes *down*.

### How the session runs

| | Part | What happens |
|---|---|---|
| **1** | 🎤 **Assignment 2 on screen — 60 min** | **The hour opens with presentations of last session's work.** Two speakers per group, 5+ minutes, then two of questions. |
| **2** | 🎬 **Demo — 45 min** | Block A. The instructor walks it; you watch. Do not type along — you keep this file. |
| **3** | 📋 **Pick a topic** | Your group claims one of the ten. First come, first served. |
| **4** | 🟠 **Your hour — 60 min** | The section at the bottom. The same five steps as before — with no target column. |

---
## 🎤 The 45 minutes we actually walk through

**The demo half of this notebook holds more than the slot.** The five below are the ones we walk together.
Everything else is reference you keep. *(This table is a map, not something read aloud.)*

| | Walked in the demo | Why this one earns the time |
|:--:|---|---|
| 1 | **How the session runs** | so the hour is not spent guessing what to hand in |
| 2 | **A.1** — k-Means, and a picture for every k | the algorithm from Lecture 5, and **why it always answers** |
| 3 | **A.2** — which k, and a hidden answer key | elbow · silhouette 🆕 · then `species` comes out of hiding: **68% right** |
| 4 | **A.3** — the scale decides the answer | the same model on the same rows goes to **92%** — and the silhouette *drops* |
| 5 | **Part 2 — picking your topic** | you skim and claim, not read out loud |

**B, C and the 📖 cell after A.3 are yours to read** — about 12 minutes if you sit down with them.
**B** is the shape k-Means cannot see and the other method in Lecture 5's outline; **C** is turning
clusters into something a manager can read.

> **No target column this session.** Every one of the ten topics is a clustering question.

---
---
# 🎬 Part 1 — The Demo · block A

**Watch, do not type along.** The header of each subsection says whether it is walked live (🎤)
or reference (📖).

---
# A · Groups Nobody Labelled
🎤 **Walked live: all of A.** Run the lecture's algorithm, get a clean answer — then find out what
the clean picture was hiding.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_moons

SEED = 42
np.random.seed(SEED)
pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('ready')

### 🔵 A.1 — k-Means: it always answers

**344 penguins, four body measurements each** — bill length, bill depth, flipper length, body mass.
The table also has a `species` column. **We hide it.** For the next two subsections the model sees
only the four numbers, exactly the situation your own topic puts you in: rows, no labels.

In [ ]:
pen  = sns.load_dataset('penguins')
COLS = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

df = pen.dropna(subset=COLS).reset_index(drop=True)   # 2 rows have no measurements at all
X  = df[COLS]
species = df['species']            # the answer key. Hidden until A.2 - the model never sees it.

print('rows with all four measurements :', len(df), 'of', len(pen))
print()
print(X.head(3).to_string())
print()
print('spread of each column (std):')
print(X.std(ddof=0).round(1).to_string())

✅ **Expected:** `342` of 344 rows · four columns · and the spreads: `bill_length_mm` 5.5, `bill_depth_mm`
2.0, `flipper_length_mm` 14.0, **`body_mass_g` 800.8** — one column is a hundred times wider than the
others. Hold on to that; it comes back in A.3.

**k-Means, as Lecture 5 pages 30–32 describe it:** pick k centres, give every row to its nearest centre,
move each centre to the mean of its rows, repeat until nothing moves. Ask for three groups:

In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(X)
df['cluster_raw'] = km3.labels_

print('rows per cluster:', np.bincount(km3.labels_))
print()
print('centre of each cluster  (the four means):')
print(pd.DataFrame(km3.cluster_centers_, columns=COLS).round(1).to_string())

plt.figure(figsize=(7, 4.4))
for c in range(3):
    m = df['cluster_raw'] == c
    plt.scatter(df.loc[m, 'bill_length_mm'], df.loc[m, 'body_mass_g'], s=14, alpha=0.7, label='cluster %d  (%d rows)' % (c, m.sum()))
plt.scatter(km3.cluster_centers_[:, 0], km3.cluster_centers_[:, 3], marker='X', s=160, color='k', label='centres')
plt.xlabel('bill length (mm)'); plt.ylabel('body mass (g)')
plt.title('k = 3: three clean groups. Are they real?')
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

✅ **Expected:** clusters of `165`, `68` and `109` rows, three centres, and a picture that looks like an
answer — three bands stacked by body mass.

**Now ask for a different number of groups.** Same rows, same algorithm, only `k` changes:

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for a, k in zip(ax, (2, 4, 5, 6)):
    lab = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X)
    for c in range(k):
        m = lab == c
        a.scatter(df.loc[m, 'bill_length_mm'], df.loc[m, 'body_mass_g'], s=9, alpha=0.7)
    a.set_title('k = %d' % k, fontsize=10); a.set_xlabel('bill length (mm)')
ax[0].set_ylabel('body mass (g)')
fig.suptitle('Every k gets a tidy answer. k-Means never says "there are no groups here".', fontsize=10)
fig.tight_layout(); plt.show()

✅ **Expected:** four tidy pictures. Two bands, four bands, five, six — every one of them looks like a
result.

**That is the first thing to know about k-Means.** It does not test whether groups exist; it divides
the rows into however many you asked for, as neatly as it can. Lecture 5 page 32 lists it as the first
weakness — *"necessity of specifying k"* — and the second is on the same page: *"clusters are sensitive
to the initial assignment of centroids … not a deterministic algorithm"*. That is why every call above
carries `random_state=SEED` and `n_init=10` (ten random starts, keep the best).

**Before we choose k, watch the algorithm do its two moves** — on the eight points from Lecture 5
page 41, with the same starting centres the slide uses (P1, P2, P7):

In [ ]:
P = np.array([[2, 2], [1, 14], [10, 7], [1, 11], [3, 4], [11, 8], [4, 3], [12, 9]], float)   # P1 .. P8
C = P[[0, 1, 6]].copy()                                                                  # start: P1, P2, P7

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for step in range(3):
    dist = ((P[:, None, :] - C[None, :, :]) ** 2).sum(-1)     # squared Euclidean distance to each centre
    lab  = dist.argmin(axis=1)                                 # move 1: every point to its nearest centre
    for c in range(3):
        ax[step].scatter(P[lab == c, 0], P[lab == c, 1], s=60)
    ax[step].scatter(C[:, 0], C[:, 1], marker='X', s=180, color='k')
    for i, (x, y) in enumerate(P):
        ax[step].annotate('P%d' % (i + 1), (x, y), textcoords='offset points', xytext=(5, 4), fontsize=8)
    ax[step].set_title('round %d: assign to nearest centre' % (step + 1) if step < 2 else 'round 3: nothing moves', fontsize=10)
    print('round %d  assignment %s   centres %s' % (step + 1, lab.tolist(), C.round(2).tolist()))
    C = np.array([P[lab == c].mean(axis=0) for c in range(3)])  # move 2: each centre to the mean of its points
fig.tight_layout(); plt.show()

✅ **Expected:** round 1 starts with centres `[2, 2]`, `[1, 14]`, `[4, 3]` · after round 2 they are
`[2, 2]`, `[1, 12.5]`, `[8, 6.2]` · after round 3 they are `[3, 3]`, `[1, 12.5]`, `[11, 8]` and the
assignment `[0, 1, 2, 1, 0, 2, 0, 2]` no longer changes — converged.

**Two moves, repeated: assign, then re-centre.** That is the whole algorithm, and it is what `KMeans`
did on the penguins in a fraction of a second. *(One difference from the slide: page 41 measures
distance as |x₁ − x₂| + |y₁ − y₂|; `sklearn` uses straight-line distance, so the rounds can land
in slightly different places. Same two moves.)*

### 🔵 A.2 — Which k? And a hidden answer key

> **🆕 New.** Lecture 5 says k must be chosen (page 32) and that good clusters are tight inside and far
> apart (page 51), but gives no way to *measure* either. The two tools below are the standard ones.
> Neither is in a lecture deck; both are diagnostics, not models.

- **Elbow:** for each k, the total squared distance of rows to their centre — page 51's *"squared error"*,
  which `sklearn` calls `inertia_`. It always falls as k grows; you look for where it stops falling fast.
- **Silhouette:** for each row, how close it is to its own cluster compared with the nearest other one,
  averaged: **+1** means tight and far apart, **0** means on the border, **below 0** means probably in
  the wrong cluster.

In [ ]:
ks = range(2, 8)
inertia = {k: KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X).inertia_ for k in range(1, 8)}
sil     = {k: silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X)) for k in ks}

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(list(inertia), list(inertia.values()), 'o-'); ax[0].set_xlabel('k'); ax[0].set_ylabel('inertia  (total squared distance to centre)')
ax[0].set_title('Elbow: the drop slows after 2 or 3', fontsize=10)
ax[1].plot(list(sil), list(sil.values()), 'o-', color='tab:red'); ax[1].set_xlabel('k'); ax[1].set_ylabel('silhouette')
ax[1].set_title('Silhouette: highest at k = 2, then k = 3', fontsize=10)
fig.tight_layout(); plt.show()

print(pd.DataFrame({'inertia': pd.Series(inertia).round(0), 'silhouette': pd.Series(sil).round(4)}).to_string())

✅ **Expected:** inertia `219386618` at k = 1, `58696922` at k = 2, `29186589` at k = 3, then a slower slide
· silhouette `0.6296` at k = 2, `0.5767` at k = 3, `0.5545` at k = 4 and falling.

**Both tools say two or three, and neither says which.** The elbow bends at 2 and again, less, at 3; the
silhouette prefers 2 but 0.58 at 3 is still a strong score. **This is normal.** The tools narrow the
choice; the reason for the final pick has to be yours, and *"the chart bends here"* is not a reason.

We take k = 3 — three bands in the picture, and a silhouette near 0.6. **Now the one thing your own topic
will never let you do: bring out the answer key.**

In [ ]:
ct = pd.crosstab(df['cluster_raw'], species)
purity = ct.max(axis=1).sum() / len(df)

print('penguins per species (the answer key, now out of hiding):', species.value_counts().to_dict())
print()
print('cluster (rows)  vs  the species we hid:')
print(ct.to_string())
print()
print('rows whose cluster matches its majority species : %.1f%%' % (purity * 100))
print()
print('mean body mass per cluster :', df.groupby('cluster_raw')['body_mass_g'].mean().round(0).astype(int).to_dict())
print('mean bill length per cluster:', df.groupby('cluster_raw')['bill_length_mm'].mean().round(1).to_dict())

# colour = the cluster the model chose, shape = the species we hid. Where they disagree, the model was wrong.
MARK = {'Adelie': 'o', 'Chinstrap': '^', 'Gentoo': 's'}
COLR = {0: 'tab:blue', 1: 'tab:orange', 2: 'tab:green'}

def cluster_vs_truth(ax, col, title):
    for c in range(3):
        for sp, mk in MARK.items():
            m = (df[col] == c) & (species == sp)
            ax.scatter(df.loc[m, 'bill_length_mm'], df.loc[m, 'flipper_length_mm'], s=22, alpha=0.75, color=COLR[c], marker=mk)
    ax.set_title(title, fontsize=10); ax.set_xlabel('bill length (mm)')
    from matplotlib.lines import Line2D
    handles  = [Line2D([], [], ls='', marker='o', color=COLR[c], label='colour: cluster %d' % c) for c in range(3)]
    handles += [Line2D([], [], ls='', marker=mk, color='k', label='shape: %s' % sp) for sp, mk in MARK.items()]
    ax.legend(handles=handles, fontsize=7, ncol=2)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
cluster_vs_truth(ax, 'cluster_raw', 'colour = cluster (raw units), shape = species: the shapes do not follow the colours')
ax.set_ylabel('flipper length (mm)')
fig.tight_layout(); plt.show()

✅ **Expected**

| cluster | Adelie | Chinstrap | Gentoo |
|---|--:|--:|--:|
| 0 | 112 | 52 | 1 |
| 1 | 0 | 0 | 68 |
| 2 | 39 | 16 | 54 |

**68.4% of rows sit in a cluster dominated by their own species** — and the other 31.6% do not. In the
picture, **colour is the cluster the model chose and shape is the species we hid** — one shape per
species — so every place a colour band cuts across a shape is a wrong penguin. Cluster 0
mixes Adelie with three-quarters of the Chinstraps; cluster 2 is a third of everything. Look at what the
clusters *are*: mean body mass **3518 g, 4458 g, 5450 g** — three weight bands, cut at round numbers.
**The model grouped the penguins by weight and nothing else.** A silhouette of 0.58 and a clean picture,
and a third of the answer wrong. A.3 says why.

### 🔵 A.3 — The scale decides the answer

Look back at A.1's spreads: body mass varies by **800 g**, bill length by **5.5 mm**. k-Means measures
distance the way Lecture 5 page 45 defines it — *Euclidean*, the square root of the summed squared
differences — so a difference of 100 g counts a hundred times more than a difference of 1 mm. **In
those units, the other three columns barely exist.** Cluster by weight is the only answer available.

**`StandardScaler` puts every column on the same footing**: subtract the column's mean, divide by its
spread, so one unit means "one standard deviation" in every column (session 2, A.1). Then the same
k-Means, same k:

In [ ]:
Xs = StandardScaler().fit_transform(X)
km3s = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(Xs)
df['cluster_scaled'] = km3s.labels_

ct_s = pd.crosstab(df['cluster_scaled'], species)
purity_s = ct_s.max(axis=1).sum() / len(df)
print('scaled - cluster vs species:')
print(ct_s.to_string())
print()
print('rows matching their majority species : raw %.1f%%   scaled %.1f%%' % (purity * 100, purity_s * 100))
print('silhouette                           : raw %.4f   scaled %.4f' % (sil[3], silhouette_score(Xs, km3s.labels_)))
rand = lambda: np.random.RandomState(SEED).randint(0, 3, len(X))          # the same random split each time
print('silhouette of a RANDOM 3-way split   : raw %+.4f  scaled %+.4f   <- the baseline both have to beat'
      % (silhouette_score(X, rand()), silhouette_score(Xs, rand())))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), sharex=True, sharey=True)
cluster_vs_truth(ax[0], 'cluster_raw',    'raw units: colour cuts across the shapes  (68% match)')
cluster_vs_truth(ax[1], 'cluster_scaled', 'scaled: one colour per shape  (92% match)')
ax[0].set_ylabel('flipper length (mm)')
fig.tight_layout(); plt.show()

✅ **Expected**

| cluster | Adelie | Chinstrap | Gentoo |
|---|--:|--:|--:|
| 0 | 24 | 63 | 0 |
| 1 | 0 | 0 | 123 |
| 2 | 127 | 5 | 0 |

Match goes from **68.4% to 91.5%**: every Gentoo in one cluster, 127 of 151 Adelie in another, 63 of 68
Chinstrap in the third. Same rows, same algorithm, same k — **only the units changed.** Same picture as
A.2 — colour is the cluster, shape is the species — and on the right each colour now holds one shape,
in bill length against flipper length, the two columns the raw model could not hear.

**And the silhouette went the other way: 0.5767 raw, 0.4472 scaled.** The better answer scored worse.
Silhouette measures how separated the clusters are *in the units you gave it* — in raw units the weight
bands really are far apart, so it praised the wrong answer. **It cannot tell you your units were wrong;
only a look at what the clusters are can.** Both are far above the random split's `-0.0228` / `-0.0154`,
which is the line any clustering has to clear.

**The rule for your own topic: scale before you cluster — always — and say so in step 2.** Session 1
said trees ignore scale; this is the session where it decides the answer.

#### 📖 Read later — what a silhouette can and cannot tell you

*Not walked. Two sentences to remember when you read your own score.*

- **A silhouette compares clusters to each other in whatever space you clustered in.** Change the units,
  drop a column, take a log, and the score moves — without the data changing. So it ranks *choices of k on
  the same data*; it does not certify that the data or the units were right (A.3 just showed it praising
  the wrong answer).
- **Scores near zero, or a random split that scores nearly as well as yours, mean the structure is weak.**
  Report the random-split number beside yours, the way session 2 reported the baseline beside R².

---
# B · The shape k-Means cannot see
📖 **Reference — read on your own.**

k-Means draws a straight boundary between neighbouring centres, so it assumes every group is a round
blob. Give it two crescents and it will cut each one in half — and, as A.3 warned, **the silhouette will
still look reasonable.** Lecture 5's outline (page 29) lists one other family, *hierarchical* clustering,
which does not assume round groups because it never uses centres at all: it merges the two closest
points, then the two closest groups, and so on, and you cut the tree at the height you want.

In [ ]:
Xm, truth = make_moons(n_samples=400, noise=0.06, random_state=SEED)

lab_km  = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(Xm)
lab_agg = AgglomerativeClustering(n_clusters=2, linkage='single').fit_predict(Xm)    # merge nearest neighbours first

def agree(lab):                                          # the crescents are the truth here; label order is arbitrary
    return max((lab == truth).mean(), (lab != truth).mean())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, lab, name in [(ax[0], lab_km, 'k-Means'), (ax[1], lab_agg, 'hierarchical (single linkage)')]:
    a.scatter(Xm[:, 0], Xm[:, 1], c=lab, s=12, cmap='coolwarm')
    a.set_title('%s: silhouette %.4f, matches the crescents %.0f%%' % (name, silhouette_score(Xm, lab), agree(lab) * 100), fontsize=10)
    print('%-30s silhouette %.4f   matches the crescents %.0f%%' % (name, silhouette_score(Xm, lab), agree(lab) * 100))
fig.tight_layout(); plt.show()

✅ **Expected:** k-Means silhouette `0.4881`, matching the crescents `75%` · hierarchical silhouette `0.3255`,
matching **`100%`**.

**Read those two lines against each other.** The method that got every point right scored *lower*. The
silhouette rewards round, well-separated groups; crescents are neither, so the honest answer looks worse
on the metric. **When your topic's groups are not blobs, the metric is not the judge — the picture is.**
*(Hierarchical clustering is in Lecture 5's outline and nowhere else in the deck, so treat it as new;
`linkage='single'` means "merge whichever two groups have the closest pair of points".)*

---
# C · Naming the groups
📖 **Reference — read on your own.**

A cluster called "2" is a technical result. A cluster called *"small, rich, long-lived — mostly
European"* is something a manager can act on. The step between them is a **profile table**: the mean of
each column per cluster, read across.

142 countries in 2007 from the gapminder table that ships with `plotly`: life expectancy, income per
person, population. Income and population are skewed by orders of magnitude, so they go in as `log10`
— and, as A.3 says, everything is scaled.

In [ ]:
import plotly.express as px
gap = px.data.gapminder().query('year == 2007').reset_index(drop=True)   # one row per country, or you cluster country-years

G  = pd.DataFrame({'lifeExp': gap['lifeExp'], 'log_gdp': np.log10(gap['gdpPercap']), 'log_pop': np.log10(gap['pop'])})
Gs = StandardScaler().fit_transform(G)
gap['cluster'] = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(Gs)

print('countries:', len(gap), '  silhouette (k = 3): %.4f' % silhouette_score(Gs, gap['cluster']))
print()
profile = gap.groupby('cluster').agg(countries=('country', 'size'), life_exp=('lifeExp', 'mean'),
                                     income_median=('gdpPercap', 'median'), pop_median=('pop', 'median'))
profile['top_continents'] = gap.groupby('cluster')['continent'].agg(lambda s: ', '.join('%s %d' % kv for kv in s.value_counts().head(2).items()))
print(profile.round(1).to_string())

✅ **Expected:** `142` countries, silhouette `0.3794`, and a profile table with three rows — one cluster of
`50` countries with life expectancy about `52.9` and a median income of `1154.6` per person (Africa 42 of them),
one of `36` with life expectancy `74.3`, median income `11112.1` and populations in the tens of millions
(Asia 16, Americas 8), and one of `56` with life expectancy `74.9`, median income `12662.9` and small
populations (Europe 22, Americas 16).

**Now name them — in a sentence each, for someone who will never see the table.** *"Fifty poorer countries
with life expectancy in the fifties, most of them African"* is a name. *"Cluster 1"* is not. The last
column of your own profile table should be the sentence, and it is what step 5 of your hour is for.

---
#### 🎨 Polish, step 3 — Read later: pick the chart from the question, not from habit

Session 1: the title is the conclusion. Session 2: equal axes, a zero line, label the points that escape.
**This session's rung:** before you draw, say which of three questions the chart answers, because each
has its own chart.

| The question | The chart | This session's example |
|---|---|---|
| **How much, compared with what?** | bars, sorted | rows per cluster · the profile table as bars |
| **How is it spread out?** | histogram · box plot | body mass per cluster — do the bands overlap? |
| **Does one thing move with another?** | scatter | bill length against flipper length, coloured by cluster (A.3) |

A pairplot is the scatter question asked for every pair of columns at once — use it to *find* the pair
that separates your groups, then put **that one pair** on the slide.

---
---
# 📋 Part 2 — Pick Your Topic
### Assignment 3 · one of these 10 · this takes ~8 minutes

**Groups of three or four. One topic per group, first come first served, no two groups on the same one.**

**Every topic here is a clustering question — rows with no label.** Take yours and go the whole way:

```
your data -> EDA -> scale (Pipeline) -> cluster_baseline() -> k-Means -> a k you can justify -> what the groups are
```

📄 **[How it is marked, and what to hand in](https://classes.incortx.com/DataAnalytics/session-00/)** — the short version: you need **a baseline beside
every number**, **scaling inside a `Pipeline`**, and **a reason for k that is not "the chart bends here"**.
Groups that turn out not to exist still score full marks if you can show why.

**The 10 topics.** 🟢 straightforward · 🟡 has something awkward in it

The **trap** column is not a general warning. It is the specific thing that will bite you in that
dataset, and it is where the questions will come from. Several topics carry a hidden answer key the way
penguins did — use it the way A.2 did: cluster first, look second.

| # | Topic | Data | The trap | |
|:--:|---|---|---|:--:|
| **1** | Group countries by development indicators | `px.data.gapminder()` -- **filter to one year first** (`year==2007` gives 142 countries) | 1704 rows = 142 countries x 12 years. **Without filtering the year you cluster "country-years", not countries** | 🟡 |
| **2** | Do three wine regions separate on chemistry? | `sklearn.datasets.load_wine()` | 13 features on wildly different units (proline in the thousands, hue in single digits) -- the clearest example in the bank of scaling flipping the answer | 🟢 |
| **3** | Do handwritten digits group themselves? | `sklearn.datasets.load_digits()` (1797 images, 8x8) | No scale worries here, but **64 dimensions start to make distance meaningless** (curse of dimensionality) -- the clusters you get may not match the actual digits | 🟡 |
| **4** | How many kinds of wholesale customer are there? | `archive.ics.uci.edu/static/public/292/wholesale+customers.zip` | Spending is heavily right-skewed -- **without a log transform first, k-Means gives you a cluster containing one large customer and nobody else** | 🟢 |
| **5** | Do three wheat varieties separate? | `archive.ics.uci.edu/static/public/236/seeds.zip` | 210 rows, 3 perfectly balanced classes -- **easy enough that the silhouette comes out high.** The real task is explaining why some datasets are easier than others | 🟢 |
| **6** | Do tumours group themselves? | `sklearn.datasets.load_breast_cancer()` | The 30 columns are **10 measurements seen 3 ways**, so they correlate strongly -- distance counts the same evidence three times and the clusters tilt towards whatever is repeated most | 🟡 |
| **7** | Three iris species -- the classic everyone should do once | `sklearn.datasets.load_iris()` | **Two of the three overlap**, so k-Means cannot separate them at any k -- a good demonstration that a high silhouette does not mean a correct answer | 🟢 |
| **8** | Do phone sensor activities group themselves? | `archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip` | **561 features** -- distance in high dimensions starts to mean nothing, so you must reduce or select first | 🟡 |
| **9** | How do heart patients group? | `archive.ics.uci.edu/static/public/45/heart+disease.zip` | Missing values are written as **`?`**, and there are several files from several hospitals -- **you have to choose which file and say why** | 🟡 |
| **10** | What do cars actually group by? | `sns.load_dataset('mpg')` | `origin` is there as an answer key to lay over the top -- **but the groups k-Means finds usually split on engine size rather than country**, which is the more interesting answer | 🟢 |

In [ ]:
# ── Record your group's choice ─────────────────────────────────────────────
TOPIC_ID = None      # <- put your group's topic number here, then run this cell

TOPIC_TRAPS = {
     1: ('Group countries by development indicators',
        '1704 rows = 142 countries x 12 years. **Without filtering the year you cluster "country-years", not countries**'),
     2: ('Do three wine regions separate on chemistry?',
        '13 features on wildly different units (proline in the thousands, hue in single digits) -- the clearest example in the bank of scaling flipping the answer'),
     3: ('Do handwritten digits group themselves?',
        'No scale worries here, but **64 dimensions start to make distance meaningless** (curse of dimensionality) -- the clusters you get may not match the actual digits'),
     4: ('How many kinds of wholesale customer are there?',
        'Spending is heavily right-skewed -- **without a log transform first, k-Means gives you a cluster containing one large customer and nobody else**'),
     5: ('Do three wheat varieties separate?',
        '210 rows, 3 perfectly balanced classes -- **easy enough that the silhouette comes out high.** The real task is explaining why some datasets are easier than others'),
     6: ('Do tumours group themselves?',
        'The 30 columns are **10 measurements seen 3 ways**, so they correlate strongly -- distance counts the same evidence three times and the clusters tilt towards whatever is repeated most'),
     7: ('Three iris species -- the classic everyone should do once',
        '**Two of the three overlap**, so k-Means cannot separate them at any k -- a good demonstration that a high silhouette does not mean a correct answer'),
     8: ('Do phone sensor activities group themselves?',
        '**561 features** -- distance in high dimensions starts to mean nothing, so you must reduce or select first'),
     9: ('How do heart patients group?',
        'Missing values are written as **`?`**, and there are several files from several hospitals -- **you have to choose which file and say why**'),
    10: ('What do cars actually group by?',
        '`origin` is there as an answer key to lay over the top -- **but the groups k-Means finds usually split on engine size rather than country**, which is the more interesting answer'),
}

if TOPIC_ID in TOPIC_TRAPS:
    title, trap = TOPIC_TRAPS[TOPIC_ID]
    print('Topic %d: %s' % (TOPIC_ID, title))
    print('Watch out for : %s' % trap)
else:
    print('Set TOPIC_ID to your group number (1-10) and run this cell again.')

✅ **Expected:** your topic and its trap printed back at you. Write the trap somewhere you will see it
again — it is the first thing to check when your results look strange.

---
---
# 🟠 Part 3 — Your Hour · 60 Minutes, Your Own Data

**Everything above was the demo.** From here it is your group's work, and it is what gets marked.

This section does not depend on a single cell above it. Run it from the top of this section and it works.

### The steps, and the clock

| Minutes | Step | What has to exist when you are done |
|:--:|---|---|
| — | **0 · Data already loaded** | you did this before class. `shape` · `head()` · one sentence on what a row is |
| 0–12 | **1 · EDA → one insight** | a chart, and a sentence stating what you *found* |
| 12–27 | **2 · Prepare the data** | what was wrong, what you did, **and why that choice** — scaling included, inside a `Pipeline` |
| 27–37 | **3 · Metric + baseline** | silhouette **with the random split beside it** — `cluster_baseline()` |
| 37–50 | **4 · Today's technique** *(if you get there)* | k-Means for several k, a k you can defend, and what the groups *are* |
| 50–60 | **5 · Write up + get ready** | one sentence per group, one limitation, notebook scrolled to where you start |

**Steps 1, 2 and 3 are what you present and what is marked. Step 4 is a bonus.**

**There is no `y` this session, so there is no train/test split** — nothing to hold out. Scaling still
goes inside a `Pipeline`, so it can be reused on new rows the same way.

In [ ]:
# ── SUBMISSION HEADER — fill this in first ─────────────────────────────────
GROUP     = ''            # your group letter: 'A' .. 'J'
MEMBERS   = ['', '', '']  # everyone in the group - keep this order all term
TOPIC_ID  = None          # the topic number your group claimed

# EVERY member speaks in the video. Every assignment, no exceptions.
IN_CLASS  = ['', '']      # the TWO representing the group in the room

# ── check ───────────────────────────────────────────────────────────────────
_all   = [m.strip() for m in MEMBERS  if m.strip()]
_room  = [m.strip() for m in IN_CLASS if m.strip()]

print(f'Group {GROUP or "?"} | topic {TOPIC_ID} | {len(_all)} members')
print(f'  in the room  : {", ".join(_room) or "-- nobody --"}')
print(f'  in the video : everyone - {", ".join(_all) or "-- nobody --"}')

unknown = [m for m in _room if m not in _all]
if not GROUP or not _all or TOPIC_ID is None:
    print('\n[ ] header not filled in yet')
elif unknown:
    print(f'\n[!] not found in MEMBERS: {", ".join(unknown)} - check the spelling')
elif len(_room) != 2:
    print(f'\n[!] {len(_room)} named for the room, should be exactly 2')
else:
    print('\n[ok] two representatives named, and every member speaks in the video')

### 🟠 Setup for this section

Its own imports and its own helpers, so this half runs whatever happened above.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import io as _io, zipfile, urllib.request      # several topics are zips this session
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine, load_iris, load_digits, load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEED = 42
np.random.seed(SEED)
pd.set_option('display.width', 120); pd.set_option('display.max_columns', 30)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('ready')

**Two helpers come with the notebook: `eda()` and `cluster_baseline()`.** Run the cell below once.

**Neither does the thinking.** `eda()` prints the six views you should always look at;
`cluster_baseline()` prints the silhouette of a random split next to yours. **Reading them is the marked
part.**

In [ ]:
def eda(df, target=None, n=6):
    """Print the six things you should always look at first. Reading them is your job.

        eda(df)                    # no target column yet
        eda(df, target='outcome')  # adds class balance + correlation with the target
    """
    import pandas as _pd
    line = '\u2500' * 62

    print(line); print(f'1. SHAPE      {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(line); print('2. ONE ROW    what does a single row actually represent?')
    print(df.head(3).to_string())

    print(line); print('3. TYPES      a number stored as text will not go into a model')
    info = _pd.DataFrame({'dtype': df.dtypes.astype(str),
                          'non_null': df.notna().sum(),
                          'distinct': df.nunique()})
    print(info.to_string())

    print(line); print('4. MISSING    how much, and in which columns')
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if len(miss) == 0:
        print('   isna() finds none  --  but a 0 or a -200 can be a missing value in disguise.')
        print('   Check step 5 for impossible values before you believe this.')
    else:
        print(_pd.DataFrame({'missing': miss, 'percent': (miss / len(df) * 100).round(1)}).to_string())

    print(line); print('5. RANGES     look for a min or max that cannot be real')
    num = df.select_dtypes('number')
    print(num.describe().T[['min', '25%', '50%', '75%', 'max']].to_string() if len(num.columns)
          else '   no numeric columns')

    if target is not None and target in df.columns:
        print(line); print(f'6. TARGET     {target!r}')
        y = df[target]
        if y.dtype.kind == 'f' or y.nunique() > 20:
            print(y.describe().to_string())
            corr = num.corr(numeric_only=True)[target].drop(target).sort_values(key=abs, ascending=False)
            print(f'\n   strongest correlations with {target}:')
            print(corr.head(n).round(4).to_string())
        else:
            print(y.value_counts(normalize=True).round(4).to_string())
            print(f'\n   most common class is {y.value_counts(normalize=True).max():.1%} of rows')
    print(line)
    print('Now write ONE sentence about something you did not know 60 seconds ago.')

def cluster_baseline(X, labels, seed=42):
    """Print the silhouette of YOUR clusters next to the silhouette of a random split.

    You do not have to write a baseline. You do have to read it and say what it
    means -- that is the part that carries marks.

        cluster_baseline(X_scaled, model.labels_)
    """
    import numpy as _np
    from sklearn.metrics import silhouette_score
    labels = _np.asarray(labels)
    k = len(_np.unique(labels))
    yours = silhouette_score(X, labels)
    rng = _np.random.RandomState(seed)
    rand = silhouette_score(X, rng.randint(0, k, len(labels)))
    print(f'baseline = split the rows into {k} groups at random')
    print(f'  silhouette, random split : {rand:+.4f}   <- at or just below zero is what random looks like')
    print(f'  silhouette, your clusters: {yours:+.4f}')
    print('  read it as: how far above "random" did the structure you found get you')
    return {'yours': yours, 'random': rand, 'k': k}

print('eda() and cluster_baseline() ready')

### 🟠 Opening a zip, if your topic is one

**Five of the ten topics are zips.** You should have loaded yours before class, but here is the shape in
one place.

```python
with urllib.request.urlopen(URL) as response:
    z = zipfile.ZipFile(_io.BytesIO(response.read()))
print(z.namelist())                         # always look before you read
df = pd.read_csv(z.open('the_file.csv'), sep=';')
```

Topic 5 (seeds) is whitespace-separated with no header: `pd.read_csv(f, sep=r'\s+', header=None)`.
Topic 9 (heart disease) writes missing values as `?`: `pd.read_csv(f, na_values='?')`.
Session 1's block B has the full four-shapes reference if you need it.

### 🟠 Step 0 — Your data, already loaded

Load it and run `eda()` straight away. There is no target column, so call it with no `target` — view 5,
the ranges, is the one to read: **which columns are a hundred times wider than the others?**

In [ ]:
# TODO: load your group's data into `df`, then run eda() on it

df = None
# eda(df)

✅ **What topic 2's `eda()` hands you in under a second**

- **view 1: 178 rows × 13 columns**, and view 4 finds nothing missing.
- **view 5 is the whole story:** `proline` runs from `278` to `1680`, `magnesium` from `70` to `162`,
  while `hue` sits between `0.48` and `1.71`. One column is a thousand times wider than another.

**Write the one sentence:** what is one row of this data?

### 🟠 Step 1 — EDA → one insight · *0–12 min*

**The chart is not the deliverable. The sentence under it is.**

With no target, the chart that earns its place is the one that shows **whether the columns are on
comparable scales** (a box plot of every column, raw) — because A.3 showed that this decides the
answer — and, if the table is small enough, a **pairplot** to see whether any pair of columns already
shows separate lumps.

| Not an insight | An insight |
|---|---|
| "This is a box plot of the columns." | "`proline` is a thousand times wider than `hue`; unscaled, distance is proline and nothing else." |
| "There are three clusters." | "Bill length against flipper length shows three lumps by eye — k = 3 has a reason before any score." |

In [ ]:
# TODO: one chart that shows something about your columns

✅ **Expected on topic 2:** `proline` alone holds **99.8%** of the total variance. Unscaled, k-Means on
this table is k-Means on one column.

**What I found:** *(one sentence — a finding, not a description of the chart)*

**What this changes about what I do next:** *(write it here)*

### 🟠 Step 2 — Prepare the data · *12–27 min*

Everything step 1 told you was wrong, you now fix — **and you write down why you fixed it that way.**

**Two hard rules, and breaking either one costs you:**

1. **Scale, inside a `Pipeline`.** A.3 is the reason. If a column is a count or an amount that runs over
   orders of magnitude (population, income, spending), take its `log10` first, then scale — C did this.
2. **Drop the answer key before clustering, if your topic has one** (`species`, `target`, `origin`). The
   model must not see it; you bring it back in step 4 to check, never to fit.

> **No `train_test_split` this session.** There is nothing to hold out, because there is no answer to
> predict. The `Pipeline` still matters: it is what you would apply to next year's rows.

In [ ]:
# TODO: build the Pipeline that prepares your columns (scale at least), and fit it

**What I fixed, and why I chose that fix:** *(one line per decision)*

### 🟠 Step 3 — Metric + baseline · *27–37 min*

**The metric this session is the silhouette** (A.2), and the baseline is the silhouette of **a random
split into the same number of groups** — `cluster_baseline()` prints both. A random split lands at or
just below zero; your clusters have to sit clearly above it.

**Two rules that still belong to you:**

1. **Call it on the same scaled matrix you clustered.**
2. **From here on, the random-split number goes next to every silhouette you print.**

**Then read it** — and remember A.3: a silhouette compares clusters *in the units you chose*. It ranks
values of k on the same data; it does not certify the data.

In [ ]:
# TODO: cluster once (k of your choice), then call cluster_baseline() on the same scaled matrix

✅ **Expected on topic 2:** random split `-0.0153` · your clusters `+0.2849`

**Our metric is ___ because ___** *(fill this in — it is half of what requirement 1 looks for)*

### 🟠 Step 4 — Today's technique · *37–50 min* — **if you get there**

**This step is a bonus, not a requirement.** Run k-Means for several k, put the silhouettes in one table
**with the random split beside them**, choose a k **for a reason**, then say what the groups *are* — the
profile table from C. If your topic has a hidden answer key, this is the only place it comes out, and
only to check.

In [ ]:
# TODO: k-Means for k = 2..6, choose one k with a reason, then describe the groups

✅ **Expected on topic 2**

| k | silhouette | random split |
|--:|--:|--:|
| 2 | 0.2593 | … |
| 3 | **0.2849** | −0.0153 |
| 4 | 0.2602 | … |

k = 3 wins on the score *and* had a reason before the score (step 1's lumps). Bring out the answer key
and **96.6%** of rows sit with their own wine class. **Without scaling the same call gets 70.2%** — with a
*higher* silhouette, `0.5711`, because in raw units the three proline bands really are far apart. A.3,
again, on your own data: **the score can praise the wrong answer; only the profile tells you which
answer you have.**

### 🟠 Step 5 — Write it up and close the file properly · *50–60 min*

Two sentences per group, both short. **You are not presenting today**, so the rest of this goes into
making the file usable when you reopen it at home:

1. **Name each cluster in one sentence** for someone who will never see the table (C is the model).
2. **Say where you got stuck and what you had already ruled out.** An unfinished step costs nothing;
   an unfinished step you cannot describe costs the write-up mark.

**What we found:** *(one sentence per group — what is it, in words a manager would use?)*

**What we do not trust:** *(one limitation — the units, the k, or whether the groups exist at all)*

**Where we got stuck:** *(if a step defeated you, say which and why — this is worth writing down)*

---
## 🟠 After the hour — presenting, and handing in

**Everything about how this is presented, questioned and marked lives in one place, and it is not this
notebook:**

📄 **[How the assignment works, and what to hand in](https://classes.incortx.com/DataAnalytics/session-00/)**

That page is the only version — it covers the minutes on screen, the code questions, asking
questions while other groups present, and every requirement for the hand-in. **Read it once at the
start of term, and again before your first turn.**

The three dates you need, and nothing else:

| When | What |
|---|---|
| **Before you leave today** | this notebook, as a file — *File → Download → Download .ipynb* |
| **Two days before session 4** | the homework: notebook, slides, video, README — **as files, no links** |
| **The start of session 4** | you present, from the notebook you handed in |

---
# Session 3 Wrap-Up

### Four things to remember

1. **k-Means always answers.** It divides the rows into k groups however many you ask for — the clean
   picture is not evidence that the groups exist.
2. **k is your choice, and the tools only narrow it.** Elbow and silhouette said "two or three"; the
   reason for the final pick was ours.
3. **Scale before you cluster.** The same model on the same rows went from 68% right to 92% right when the
   columns were put on one footing — and the silhouette went *down* while it did.
4. **A cluster is not a result until it has a name.** The profile table, read across, is what turns
   "cluster 2" into something a manager can use.

### What this session's notebook should end up carrying
- **scaling inside a `Pipeline`**, and a line saying why
- a silhouette **with the random split printed beside it**, for every k you tried
- **a name for every group**, and a k you chose for a reason

*(Dates, file formats and everything else about handing in: see the section above.)*

### Next session — the machine picks its own features
Three sessions of telling the model which columns to look at. **Next time it decides for itself** — a
neural network on the titanic table from session 1, and then on images, where there are no columns at all.